In [120]:
import pandas as pd
import numpy as np
from dateutil import parser

<h2> Reading the dataset </h2>

In [92]:
# Reading other dataset
Data1 = pd.read_excel('/Users/briankimanzi/Documents/programming Languages/PythonProgramming/JupyterNoteBook/Datasets/MpesaStatement_Katra.xlsx')
Data2 = pd.read_excel('/Users/briankimanzi/Documents/programming Languages/PythonProgramming/JupyterNoteBook/Datasets/MPESA_Marylyne.xlsx')

<h2> Cleaning the dataset </h2>

In [136]:
Data1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 135 entries, 0 to 134
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Receipt No.         135 non-null    object        
 1   Completion Time     135 non-null    datetime64[ns]
 2   Details             135 non-null    object        
 3   Transaction Status  135 non-null    object        
 4   Paid In             31 non-null     float64       
 5   Withdrawn           104 non-null    float64       
 6   Balance             135 non-null    float64       
dtypes: datetime64[ns](1), float64(3), object(3)
memory usage: 7.5+ KB


In [112]:
# Anonymising the dataset 
def transaction(x):
    x = str(x).strip()
    if x.startswith('Merchant Payment'):
        index = x.find(' - ')+2
        name = x[index:].strip().upper()
        
        return 'BUY GOODS', name

    elif x.startswith('Deposit of Funds'):
        index = x.find(' - ')+2
        name = x[index:].strip().upper()
        
        return 'AGENT DEPOSIT', name

    elif x.startswith('OD Loan Repayment'):
        return 'FULIZA REPAYMENT', 'FULIZA'

    elif x.startswith('OverDraft of Credit Party'):
        return 'FULIZA TAKEN', 'FULUZA'

    elif x.startswith('M-Shwari Deposit'):
        return 'M-SHWARI DEPOSIT FROM M-PESA', 'M-SHWARI'
        
    elif x.startswith('KCB M-PESA Deposit'):
        return 'KCB M-PESA DEPOSIT FROM M-PESA', 'KCB DEPOSIT'

    elif x.startswith('KCB M-PESA Withdraw'):
        return 'KCB M-PESA WITHDRAW FROM M-PESA', 'KCB WITHDRAW'

    elif x.startswith('Customer Transfer'):
        index = x.find(' - ')
        to_search = x[index:]
        last = 1

        for i in range(len(to_search)):
            try:
                int(to_search[i])
                last = i
                
            except:
                v = 10

        name = x[index+last+2:].strip().upper()
        return 'SEND MONEY', name

    elif x.startswith('M-Shwari Withdraw'):
        
        return 'M-SHWARI WITHDRAW FROM M-PESA', 'M-SHWARI WITHDRAW'

    elif x.startswith('Pay Bill Charge'):
        # if x.strip() == 'Pay Bill Charge':
            return 'PAY BILL CHARGES', 'TRANSACTION COST'
        
    elif x.startswith('Pay Bill Online to'):
        if x == 'Pay Bill Online to':  
            return 'PAY BILL ONLINE', 'TRANSACTION COST'
        else:
            start = len('Pay Bill Online to')+8
            end = x.lower().find('acc')
            name = x[start:end].strip().upper()
            return 'PAY BILL', name
    elif x.startswith('Pay Bill'):
        if x == 'Pay Bill':  
            return 'PAY BILL', 'UNKNOWN'
        else:
            start = x.find(' - ') + 3 if ' - ' in x else len('Pay Bill') + 1
            end = x.lower().find('acc')
            name = x[start:end].strip().upper() if end != -1 else x[start:].strip().upper()
            return 'PAY BILL', name

    elif x.startswith('Funds received'):
        index = x.find(' - ')
        to_search = x[index:]
        last = 1

        for i in range(len(to_search)):
            try:
                int(to_search[i])
                last = i
            except:
                v = 10

        name = x[index+last+2:].strip().upper()
        
        return 'RECEIVED FUNDS', name

    elif x.startswith('Customer Payment to Small Business') or x.startswith('Customer Send Money'):
        index = x.find(' - ')
        to_search = x[index:]
        last = 1

        for i in range(len(to_search)):
            try:
                int(to_search[i])
                last = i
            except:
                v = 10

        name = x[index+last+2:].strip().upper()
        
        return 'POCHI LA BIASHARA', name

    elif x.startswith('Airtime Purchase'):
        return 'AIRTIME PURCHASE', 'AIRTIME'

    elif x.startswith('Business Payment From'):
        index = x.find(' - ')
        end = x.lower().find('via')
        to_search = x[index:]
        last = 1
        name = x[index:end].strip().upper()
        return 'FUNDS RECEIVED FROM BUSINESS', name

    elif x.startswith('Customer Transfer of Funds Charge'):
        return 'TRANSACTION COST', 'TRANSACTION COST'
    
    elif x.startswith('Buy Bundles Online'):
        return 'BUNDLES PURCHASE', 'BUNDES PURCHASE'
    
    elif x.startswith('Customer Withdrawal'):
        index = x.find(' - ')+2
        name = x[index:].strip().upper()
        return 'CASH WITHDRAWAL', name
    
    elif x.startswith('Withdrawal Charge'):
        return 'CASH WITHDRAWAL CHARGES', "TRANSACTION COST"

    elif x.startswith('Promotion Payment'):
        return 'PROMOTION PAYMENT', 'M-PESA OFFERS. VIA API'
    
    elif x.startswith('Business Payment'):
        return 'BUSINESS PAYMENT', 'CO-OP BANK VIA API'
        
    elif x.startswith('Savings Contribution'):
        return 'TO HUSTLER FUND SAVINGS', 'HUSTLER FUND'
    
    elif x.startswith('Term Loan Disbursement for H- Fund') or x.startswith('Term Loan Disbursement for H-Fund'):
        return 'HUSTLER FUND Disbursement'.upper(), 'HUSTLER FUND'
    
    elif x.startswith('Term Loan Repayment for H- Fund') or x.startswith('Term Loan Repayment for H-Fund'):
        return 'HUSTLER FUND REPAYMENT', 'HUSTLER FUND'
    
    else:
        return 'UNIDENTIFIED', 'UNIDENTIFIED'

In [8]:
Data1['Details'] = Data1['Details'].apply(lambda x: x.replace('\r', ' '))

In [113]:
details = list(Data1['Details'].apply(lambda x: transaction(x)).values)
transactionType = [i[0] for i in details]
TransactionParty = [i[1] for i in details]

In [139]:
# changing the date format
def change_date(date):
    date_str = str(date)
    date = parser.parse(date_str)
    return (date.year, date.month, date.day, date.weekday(),date.hour, date.minute, date.second)

In [140]:
date = Data1['Completion Time'].apply(lambda x: change_date(x)).values
Year = [i[0] for i in date]
Month = [i[1] for i in date]
Date = [i[2] for i in date]
Weekday = [i[3] for i in date]
Hour = [i[4] for i in date]
Minute = [i[5] for i in date]
Seconds = [i[6] for i in date]

transactionDay = []

reverseDay = Date.copy()[::-1]
datey = reverseDay[0]

d = 1

for i in reverseDay:
    if i == datey:
        transactionDay.append(d)
    else:
        datey = i
        d += 1
        transactionDay.append(d)

transactionDay.reverse()

In [145]:
Date

[7,
 4,
 30,
 30,
 28,
 28,
 25,
 22,
 22,
 20,
 19,
 19,
 19,
 18,
 18,
 18,
 18,
 16,
 14,
 14,
 7,
 5,
 4,
 4,
 4,
 1,
 1,
 1,
 25,
 25,
 24,
 22,
 21,
 21,
 21,
 21,
 21,
 21,
 19,
 19,
 19,
 19,
 19,
 17,
 15,
 11,
 11,
 10,
 6,
 6,
 4,
 4,
 30,
 30,
 29,
 29,
 29,
 29,
 29,
 28,
 26,
 23,
 22,
 22,
 18,
 15,
 10,
 9,
 7,
 5,
 5,
 4,
 4,
 4,
 4,
 3,
 28,
 28,
 19,
 19,
 17,
 17,
 17,
 17,
 3,
 2,
 1,
 30,
 30,
 30,
 26,
 25,
 25,
 25,
 25,
 25,
 25,
 22,
 21,
 21,
 20,
 20,
 19,
 16,
 16,
 15,
 15,
 15,
 12,
 9,
 9,
 1,
 1,
 30,
 30,
 25,
 24,
 24,
 24,
 24,
 24,
 23,
 23,
 22,
 22,
 17,
 17,
 16,
 16,
 14,
 12,
 12,
 11,
 11,
 11]